# EvoHash parallel around config optimizer

A cleaner parallel optimizer notebook for local categorical sweeps around known-good configs.

Supported hashes: `pdq`, `phash`, `photodna`.

This is the "around" variant of notebook 37: categorical integer controls can use specs like `{'mode': 'around', 'center': 20, 'radius': 1, 'step': 10}`. The optimizer still evaluates candidates through `evaluate_attack(...)`, with parallelism only across independent image pairs.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

# Keep native math libraries from oversubscribing when process parallelism is enabled.
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

def find_project_root() -> Path:
    """Walk up from cwd to the repo folder (the one holding optimizer.py)."""
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / 'optimizer.py').is_file() and (d / 'stage2_optimize').is_dir():
            return d
    raise RuntimeError(f'Project root not found above {Path.cwd()}')

PROJECT_ROOT = find_project_root()
PACKAGE_PARENT = PROJECT_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

import evohash.optimizer as opt
opt = importlib.reload(opt)

all_trials_dataframe = opt.all_trials_dataframe
calibrate_photodna_process_workers = opt.calibrate_photodna_process_workers
comparison_dataframe = opt.comparison_dataframe
load_attack_config = opt.load_attack_config
print_tuned_configs = opt.print_tuned_configs
resolve_parallel_runtime = opt.resolve_parallel_runtime
run_hash_tuning_experiment = opt.run_hash_tuning_experiment
tuned_configs_dict = opt.tuned_configs_dict
tuning_plan_dataframe = opt.tuning_plan_dataframe

from evohash.dataset import Dataset
from evohash.optimizer import OptimizerSettings

print(f'Loaded optimizer from: {opt.__file__}')
print(f'Project root: {PROJECT_ROOT}')


## Run target

Edit `HASH_ID` and `TUNE_ATTACKS_SETTING`. `"all"` means every configured attack for the selected hash.


In [ ]:
HASH_ID = 'phash'  # 'pdq', 'phash', 'photodna'

# 'all' or a list like ['simba', 'atkscopes'].
TUNE_ATTACKS_SETTING = 'all'
# TUNE_ATTACKS_SETTING = ['simba', 'atkscopes']

# Smoke -> full switch.
MAX_PAIRS = 30       # smoke: 3-5; current target: 30; full zip: None
BUDGET = 20_000
N_TRIALS = 500
SEED = 0
SPLIT = 'public'

CONFIG_PATH = PROJECT_ROOT / 'results' / 'attack_configs_optimized.json'
ZIP_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'dataset_v1_public.zip',
    PROJECT_ROOT.parent / 'dataset_v1_public.zip',  # legacy: next to the repo folder
]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), ZIP_CANDIDATES[0])

OUT_DIR = PROJECT_ROOT / 'outputs' / f'parallel_around_config_optimizer_{HASH_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
STORAGE_PATH = OUT_DIR / f'{HASH_ID}_optuna.sqlite3'
STORAGE = f'sqlite:///{STORAGE_PATH.as_posix()}'

if HASH_ID not in {'pdq', 'phash', 'photodna'}:
    raise ValueError("HASH_ID must be one of: 'pdq', 'phash', 'photodna'")

config = load_attack_config(CONFIG_PATH)
available_attacks = list(config['configs'][HASH_ID].keys())
if TUNE_ATTACKS_SETTING == 'all':
    TUNE_ATTACKS = available_attacks
else:
    TUNE_ATTACKS = list(TUNE_ATTACKS_SETTING)

missing = [attack for attack in TUNE_ATTACKS if attack not in available_attacks]
if missing:
    raise ValueError(f'Unknown attacks for {HASH_ID}: {missing}. Available: {available_attacks}')

PAIR_COUNT = len(Dataset(str(ZIP_PATH), split=SPLIT, max_pairs=MAX_PAIRS))

print('HASH_ID:', HASH_ID)
print('TUNE_ATTACKS:', TUNE_ATTACKS)
print('ZIP_PATH:', ZIP_PATH, 'exists:', ZIP_PATH.exists())
print('PAIR_COUNT:', PAIR_COUNT)
print('BUDGET:', BUDGET, 'N_TRIALS:', N_TRIALS)
print('OUT_DIR:', OUT_DIR)
print('STORAGE:', STORAGE)


## Parallel settings

`auto` behavior:

- Linux `pdq/phash`: `process + fork`, up to `30` pair workers.
- Windows `pdq/phash`: `thread`.
- `photodna`: `process`; optional calibration can discover a stable cap for the current Windows machine.

For PhotoDNA, set `CALIBRATE_PHOTODNA = True` once at the start of the session.


In [ ]:
PARALLEL_BACKEND = 'auto'  # 'auto', 'sequential', 'thread', 'process'
PARALLEL_WORKERS = None    # None lets optimizer resolve from hash/platform/pair count
PARALLEL_WORKER_CAP = 30
PARALLEL_MIN_PAIRS = 2
PARALLEL_SHOW_PROGRESS = False

# PhotoDNA only. If None and CALIBRATE_PHOTODNA=False, optimizer auto/default/env is used.
PHOTODNA_WORKER_CAP = None
CALIBRATE_PHOTODNA = (HASH_ID == 'photodna')
PHOTODNA_CALIBRATION_BUDGET = 100
PHOTODNA_CALIBRATION_PAIRS = min(30, PAIR_COUNT)
PHOTODNA_CALIBRATION_CAPS = None  # None => try 1..cpu_count for <=16 cores

if CALIBRATE_PHOTODNA:
    calibration = calibrate_photodna_process_workers(
        CONFIG_PATH,
        attack_id='nes',
        n_pairs=PHOTODNA_CALIBRATION_PAIRS,
        budget=PHOTODNA_CALIBRATION_BUDGET,
        seed=SEED,
        caps=PHOTODNA_CALIBRATION_CAPS,
        stop_after_first_failure=True,
        set_env=True,
        verbose=True,
    )
    PHOTODNA_WORKER_CAP = calibration.recommended_workers
    calibration_df = calibration.to_dataframe()
    calibration_path = OUT_DIR / 'photodna_worker_calibration.csv'
    calibration_df.to_csv(calibration_path, index=False, encoding='utf-8')
    print('PhotoDNA worker cap:', PHOTODNA_WORKER_CAP)
    print('Saved calibration:', calibration_path)
else:
    calibration = None

preview_settings = OptimizerSettings(
    hash_id=HASH_ID,
    attack_id=TUNE_ATTACKS[0],
    budget=BUDGET,
    seed=SEED,
    parallel_backend=PARALLEL_BACKEND,
    parallel_workers=PARALLEL_WORKERS,
    parallel_worker_cap=PARALLEL_WORKER_CAP,
    parallel_photodna_worker_cap=PHOTODNA_WORKER_CAP,
    parallel_min_pairs=PARALLEL_MIN_PAIRS,
    parallel_show_progress=PARALLEL_SHOW_PROGRESS,
)
runtime = resolve_parallel_runtime(preview_settings, PAIR_COUNT)
print('Resolved parallel runtime:', runtime)


## Around search-space controls

Local discrete specs are resolved inside `optimizer.py` before Optuna sees them.

Useful patterns:

```python
{'mode': 'around', 'center': 20, 'radius': 1, 'step': 10, 'minimum': 10}  # [10, 20, 30]
{'mode': 'around', 'radius': 2, 'step': 1, 'minimum': 2}                 # around current baseline
```


In [ ]:
# Reusable local specs.
# Around center=20 gives [10, 20, 30] for n_samples/q.
NS_AROUND_20 = {'mode': 'around', 'center': 20, 'radius': 1, 'step': 10, 'minimum': 10}
Q_AROUND_20 = {'mode': 'around', 'center': 20, 'radius': 1, 'step': 10, 'minimum': 10}

# Around the current baseline. Example: base freq_dims=10 -> [8, 9, 10, 11, 12].
FREQ_AROUND_BASE = {'mode': 'around', 'radius': 2, 'step': 1, 'minimum': 2}
MAX_FREQ_AROUND_BASE = {'mode': 'around', 'radius': 2, 'step': 1, 'minimum': 2}

CATEGORY_CHOICES_BY_HASH = {
    'pdq': {
        'nes': {'n_samples': NS_AROUND_20},
        'prokos': {'n_samples': NS_AROUND_20, 'antithetic': [True, False], 'grayscale_noise': [True]},
        'nes_attack_v0': {'n_samples': NS_AROUND_20, 'grayscale_noise': [True]},
        'simba': {'freq_dims': FREQ_AROUND_BASE, 'stride': [1]},
        'zo_signsgd': {'q': Q_AROUND_20, 'estimator': ['forward', 'central', 'majority'], 'direction_dist': ['gaussian'], 'eval_updated_point': [True]},
        'atkscopes': {'max_freq': MAX_FREQ_AROUND_BASE, 'beta1': [0.9], 'beta2': [0.999]},
    },
    'phash': {
        'nes': {'n_samples': NS_AROUND_20},
        'prokos': {'n_samples': NS_AROUND_20, 'antithetic': [True, False], 'grayscale_noise': [True]},
        'nes_attack_v0': {'n_samples': NS_AROUND_20, 'grayscale_noise': [True]},
        'simba': {'freq_dims': FREQ_AROUND_BASE, 'stride': [1]},
        'zo_signsgd': {'q': Q_AROUND_20, 'estimator': ['forward', 'central', 'majority'], 'direction_dist': ['gaussian'], 'eval_updated_point': [True]},
        'atkscopes': {'max_freq': MAX_FREQ_AROUND_BASE, 'beta1': [0.9], 'beta2': [0.999]},
    },
    'photodna': {
        'nes': {'n_samples': NS_AROUND_20},
        'prokos': {'n_samples': NS_AROUND_20, 'antithetic': [True, False], 'grayscale_noise': [True]},
        'nes_attack_v0': {'n_samples': NS_AROUND_20, 'grayscale_noise': [True]},
        'simba': {'freq_dims': FREQ_AROUND_BASE, 'stride': [1]},
        'zo_signsgd': {'q': Q_AROUND_20, 'estimator': ['forward', 'central', 'majority'], 'direction_dist': ['gaussian'], 'eval_updated_point': [True]},
        'atkscopes': {'patch_size': [64, 128, 256], 'max_freq': MAX_FREQ_AROUND_BASE, 'beta1': [0.9], 'beta2': [0.999]},
    },
}

CATEGORY_CHOICES = CATEGORY_CHOICES_BY_HASH.get(HASH_ID, {})
NUMERIC_PARAMS = {}  # {} means tune all supported continuous params
FIXED_PARAMS = {}

plan = tuning_plan_dataframe(
    config=config,
    hash_id=HASH_ID,
    attack_ids=TUNE_ATTACKS,
    category_choices=CATEGORY_CHOICES,
    numeric_params=NUMERIC_PARAMS,
    fixed_params=FIXED_PARAMS,
)
plan_path = OUT_DIR / 'tuning_plan.csv'
plan.to_csv(plan_path, index=False, encoding='utf-8')
print('Saved:', plan_path)
display(plan)


## Run optimization

In [ ]:
results = run_hash_tuning_experiment(
    config_path=CONFIG_PATH,
    zip_path=ZIP_PATH,
    hash_id=HASH_ID,
    attack_ids=TUNE_ATTACKS,
    split=SPLIT,
    max_pairs=MAX_PAIRS,
    budget=BUDGET,
    n_trials=N_TRIALS,
    seed=SEED,
    asr_floor=0.05,
    failure_gap_penalty=0.0,
    category_choices=CATEGORY_CHOICES,
    numeric_params=NUMERIC_PARAMS,
    fixed_params=FIXED_PARAMS,
    study_prefix=f'parallel_around_{HASH_ID}_budget{BUDGET}_pairs{MAX_PAIRS or "all"}',
    storage=STORAGE,
    rerun_best=False,
    verbose=True,
    parallel_backend=PARALLEL_BACKEND,
    parallel_workers=PARALLEL_WORKERS,
    parallel_worker_cap=PARALLEL_WORKER_CAP,
    parallel_photodna_worker_cap=PHOTODNA_WORKER_CAP,
    parallel_min_pairs=PARALLEL_MIN_PAIRS,
    parallel_show_progress=PARALLEL_SHOW_PROGRESS,
)


## Results and exports

In [ ]:
comparison_df = comparison_dataframe(results)
trials_df = all_trials_dataframe(results)
configs = tuned_configs_dict(results)

comparison_path = OUT_DIR / 'comparison.csv'
trials_path = OUT_DIR / 'trials.csv'
configs_path = OUT_DIR / 'tuned_configs.json'

comparison_df.to_csv(comparison_path, index=False, encoding='utf-8')
trials_df.to_csv(trials_path, index=False, encoding='utf-8')
configs_path.write_text(json.dumps(configs, indent=2, sort_keys=True), encoding='utf-8')

print('Saved:', comparison_path)
print('Saved:', trials_path)
print('Saved:', configs_path)
display(comparison_df)
display(trials_df.head(100))


In [ ]:
print_tuned_configs(results)

best_configs = tuned_configs_dict(results)
best_configs_path = OUT_DIR / "best_configs.json"
best_configs_path.write_text(json.dumps(best_configs, indent=2, sort_keys=True), encoding="utf-8")

print("\nBest configs JSON")
print(json.dumps(best_configs, indent=2, sort_keys=True))
print(best_configs_path)